# Phase 10 — Launch the full-system Streamlit UI on Colab

Wires Phases 5 (disease) + 6 (soil) + 7 (RAG) + 8 (integration) + 9 (explainability) into a single live demo.

Run the cells in order. The last cell starts streamlit + a localtunnel and prints a public URL you can open in any browser. URL is session-bound — Colab session timeout kills it.

**Requirements:** T4 GPU runtime, HF Write token (for the private corpus + Llama-3.1-8B).

In [ ]:
# Cell 2 — setup: clone repo + install runtime deps (mirrors phase5/phase6)
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os, subprocess, sys
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], check=False)
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

_pip_packages = [
    "streamlit>=1.36",
    "transformers>=4.45",
    "accelerate>=0.34",
    "bitsandbytes>=0.44",
    "sentence-transformers>=3.0",
    "chromadb>=0.5",
    "datasets>=2.20",
    "huggingface_hub>=0.24",
    "timm>=1.0.0",
    "rembg>=2.0.59",
    "pytorch-grad-cam>=1.5",
    "pydantic>=2.7",
    "opencv-python-headless",
]
subprocess.run(["pip", "install", "--quiet", *_pip_packages], check=True)
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login (Write token; same as phase5/phase6/phase7)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 4 — verify GPU
import subprocess, torch
subprocess.run(["nvidia-smi"], check=False)
print()
assert torch.cuda.is_available(), "No CUDA — switch runtime to T4 GPU."
dev = torch.cuda.get_device_properties(0)
print(f"GPU: {dev.name}, VRAM: {dev.total_memory / 1024**3:.1f} GiB")
free, total = torch.cuda.mem_get_info()
print(f"VRAM free: {free / 1024**3:.2f} / {total / 1024**3:.2f} GiB")

## Cell 5 — start Streamlit + localtunnel, print the public URL

Streamlit runs on port 8501 in the background. `localtunnel` exposes it to a public `https://*.loca.lt` URL printed below. loca.lt shows a one-time "Continue" gate that asks for a **tunnel password** = your Colab instance's public IP; we print that too. The session dies on Colab timeout (expected for a demo).

In [ ]:
import subprocess, time, urllib.request

# (a) install localtunnel via npm (node ships with Colab)
subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)

# (b) start streamlit (background; logs to /content/streamlit.log)
streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
print("streamlit pid:", streamlit_proc.pid)

# (c) wait until streamlit is reachable on 8501 (models load lazily on first request)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8501", timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("streamlit didn't come up — see /content/streamlit.log")
print("streamlit is up on port 8501")

# (d) start localtunnel; capture URL from its stdout
lt_proc = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(30):
    line = lt_proc.stdout.readline()
    if not line:
        time.sleep(1); continue
    print(line.rstrip())
    if "loca.lt" in line:
        public_url = line.strip().split()[-1]
        break
assert public_url, "localtunnel didn't print a URL — re-run this cell."

# (e) print the tunnel password (= public IP) so the user can paste it into loca.lt's gate
ip = urllib.request.urlopen("https://loca.lt/mytunnelpassword", timeout=10).read().decode().strip()
print("\n===== OPEN THIS URL =====")
print(public_url)
print("Tunnel password (paste on the loca.lt gate):", ip)
print("==========================\n")

## Cell 6 — (optional) tail the streamlit log

If the app errors in the browser, run this to see the server-side traceback.

In [ ]:
!tail -n 80 /content/streamlit.log